# Localisation

Load and query EU5 localisation files. Useful for finding display names, checking key coverage, and understanding what the game calls things internally vs. what the player sees.

In [1]:
# Setup
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\PH\Documents\Claude\Projects\PDX Save Analyzer


In [2]:
# === CONFIGURATION ===
LOC_DIR = "game-data/eu5/localization/english"      # Dev copy
# LOC_DIR = r"C:\Program Files (x86)\Steam\steamapps\common\Europa Universalis V\game\main_menu\localization\english"  # Direct from install

In [3]:
from toolbox.localisation import load_localisation, display_name

loc = load_localisation(LOC_DIR)
print(f"Loaded {len(loc):,} localisation entries from {LOC_DIR}")

Loaded 86,507 localisation entries from game-data/eu5/localization/english


## Look Up Keys

In [4]:
# Look up specific keys
KEYS = ["WUR", "SWE", "DAN", "FRA", "ENG",
        "catholic", "sunni", "orthodox",
        "swedish", "bavarian", "french",
        "age_3_discovery", "age_4_reformation"]

print(f"{'Key':<35s} {'Display Name'}")
print("-" * 60)
for key in KEYS:
    name = display_name(loc, key)
    marker = "" if name != key else " (not found)"
    print(f"  {key:<33s} {name}{marker}")

Key                                 Display Name
------------------------------------------------------------
  WUR                               Württemberg
  SWE                               Sweden
  DAN                               Denmark
  FRA                               France
  ENG                               England
  catholic                          Catholicism
  sunni                             Sunnism
  orthodox                          Orthodoxy
  swedish                           Swedish
  bavarian                          bavarian (not found)
  french                            Francien
  age_3_discovery                   Discovery
  age_4_reformation                 Reformation


## Search Localisation

In [5]:
# Search for keys or values containing a term
SEARCH = "bavaria"           # case-insensitive
SEARCH_IN = "both"           # "keys", "values", or "both"
MAX = 30

term = SEARCH.lower()
results = []
for k, v in loc.items():
    match_key = term in k.lower()
    match_val = term in v.lower()
    if (SEARCH_IN == "keys" and match_key) or \
       (SEARCH_IN == "values" and match_val) or \
       (SEARCH_IN == "both" and (match_key or match_val)):
        results.append((k, v))

print(f"Found {len(results)} entries matching {SEARCH!r} (in {SEARCH_IN}):\n")
for k, v in results[:MAX]:
    print(f"  {k:<40s} = {v}")
if len(results) > MAX:
    print(f"  ... and {len(results) - MAX} more")

Found 32 entries matching 'bavaria' (in both):

  bavarian_brewing_expertise               = Skilled Brewers
  bavarian_brewing_expertise_desc          = Our lands are rich in the crops and fresh water required to brew beer, and our people readily enjoy this abundance. Many small breweries could grow into successful businesses, with the right support.
  bavarian_stem_duchy                      = Dreams of Unification
  bavarian_zelousness_desc                 = We must ensure our people remain united in their faith, and that they will not be divided by heretical thoughts.
  bavarian_hofkriegsrat_desc               = Establishing a council with the sole duty of advising our ruler during times of conflict will help us develop a more capable fighting force.
  bavarian_flexible_diplomats              = Practical Diplomacy
  foundation_of_bavarian_army_desc         = Only with a formidable standing army will we have the military might necessary to match our ambitions. With careful focus, we

## Statistics

In [6]:
# Localisation stats
from pathlib import Path
from collections import Counter

yml_files = sorted(Path(LOC_DIR).glob("*.yml"))
print(f"Total entries : {len(loc):,}")
print(f"Total files   : {len(yml_files)}")
print()

# Count entries per file
print(f"{'File':<50s} {'Lines':>8}")
print("-" * 62)
for f in yml_files:
    lines = sum(1 for line in f.read_text(encoding='utf-8-sig').splitlines() if line.strip() and not line.strip().startswith('#') and ':' in line)
    print(f"  {f.name:<48s} {lines:>8,}")

Total entries : 86,507
Total files   : 107

File                                                  Lines
--------------------------------------------------------------
  _achievements_l_english.yml                           153
  _debug_l_english.yml                                   97
  actions_l_english.yml                               1,136
  advances_l_english.yml                              5,456
  alerts_l_english.yml                                  376
  area_l_english.yml                                    852
  artists_l_english.yml                                 994
  auto_modifiers_l_english.yml                          114
  buildings_l_english.yml                               973
  caesar_tools_l_english.yml                             81
  casus_belli_l_english.yml                             206
  character_interactions_l_english.yml                  248
  character_l_english.yml                               171
  character_names_dynamic_l_english.yml              

## Coverage Check

Check whether a list of save keys have localisation entries.

In [7]:
# Paste save keys here to check coverage
KEYS_TO_CHECK = [
    "WUR", "SWE", "DAN", "FRA", "ENG", "MNG", "OTT",
    "catholic", "sunni", "orthodox", "hindu",
    "bavarian", "swedish", "french",
    "age_3_discovery", "age_4_reformation",
    "some_fake_key_that_wont_exist"
]

found = [k for k in KEYS_TO_CHECK if display_name(loc, k) != k]
missing = [k for k in KEYS_TO_CHECK if display_name(loc, k) == k]

print(f"Coverage: {len(found)}/{len(KEYS_TO_CHECK)} ({100*len(found)/len(KEYS_TO_CHECK):.0f}%)\n")
if missing:
    print("Missing:")
    for k in missing:
        print(f"  {k}")

Coverage: 14/17 (82%)

Missing:
  OTT
  bavarian
  some_fake_key_that_wont_exist
